**Optimization pipeline: Pandas vs Polars vs PySpark**

**Dataset:** maukerja.my Malaysia job listing <br>
**Objectives:** Compare performance between 3 choosen libraries across 4 metrics: <br>
- Execution time (s)
- Memory usage (MB)
- CPU utilization (%)
- Throughput (records/s)

<h2> 1. Installation & import libraries </h2>

In [43]:
# Install required packages
!pip install -q polars psutil pyspark

In [44]:
import os
import gc
import time
import tracemalloc
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import polars as pl
import psutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)

# Warm up psutil CPU counter
proc = psutil.Process(os.getpid())
proc.cpu_percent(interval=None)

print('✅ All packages ready.')

✅ All packages ready.


<h2> 2. Upload CSV file </h2>

In [45]:
CSV_FILE = 'maukerja_cleaned_updated (1).csv' # dataset yg dah clean

with open(CSV_FILE, 'r', encoding='utf-8') as f:
    total_records = sum(1 for line in f) - 1   # subtract header

print(f'Dataset: {CSV_FILE}')
print(f'Total records: {total_records:,}')

# Preview columns
sample = pd.read_csv(CSV_FILE)
print(f'\n Columns ({len(sample.columns)}): {list(sample.columns)}')
print('\nSample rows:')
display(sample.head(3))

Dataset: maukerja_cleaned_updated (1).csv
Total records: 12,010

 Columns (9): ['url', 'job_id', 'title', 'company', 'description', 'salary_min', 'salary_max', 'state', 'country']

Sample rows:


,url,job_id,title,company,description,salary_min,salary_max,state,country
0,https://www.maukerja.my/job?jobId=26167410&slu...,26167410,MANAGEMENT TRAINEE,LTH Group Asia Sdn Bhd,Penerangan Kerja Tanggungjawab Management Trai...,2000.0,5000.0,Sabah,Malaysia
1,https://www.maukerja.my/job?jobId=26412283&slu...,26412283,BOARDING OFFICER,BATAMINDO SHIPPING & WAREHOUSING PTE LTD,Penerangan Kerja Tanggungjawab Contact person ...,8370.0,8370.0,International,Singapore
2,https://www.maukerja.my/job?jobId=26187300&slu...,26187300,"Engineer II, Quality Engineering, APS",Entegris,Penerangan Kerja Tanggungjawab Job Title: Engi...,NaN,NaN,Kedah,Malaysia


<h2> 3. Performance metrics collection </h2>

In [46]:
all_run_records = []   # every individual run
all_metrics     = []   # averaged metrics per method


def run_benchmark(method_name, fn, n_rows_fn=None, runs=3):
    """
    Run `fn` exactly `runs` times, measuring:
      - Wall-clock execution time   (time.perf_counter)
      - Peak memory delta           (tracemalloc)
      - CPU utilisation             (psutil)
      - Throughput                  (rows / elapsed)

    Returns (averaged_metrics_dict, last_result).
    """
    print(f'\n  Running: {method_name} ({runs} runs)...')

    all_times, all_cpu, all_mem, all_tput = [], [], [], []
    result = None

    for run in range(1, runs + 1):
        gc.collect()

        tracemalloc.start()
        proc.cpu_percent(interval=None)          # reset CPU counter
        t_start = time.perf_counter()

        result = fn()

        elapsed   = time.perf_counter() - t_start
        cpu_pct   = proc.cpu_percent(interval=None)
        _, peak   = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        memory_mb = peak / 1024 ** 2

        n_rows     = n_rows_fn(result) if n_rows_fn else total_records
        throughput = n_rows / elapsed if elapsed > 0 else 0

        all_times.append(elapsed)
        all_cpu.append(cpu_pct)
        all_mem.append(memory_mb)
        all_tput.append(throughput)

        all_run_records.append({
            'method':                     method_name,
            'run':                        run,
            'time_sec':                   round(elapsed, 4),
            'cpu_percent':                round(cpu_pct, 2),
            'memory_mb':                  round(memory_mb, 2),
            'throughput_records_per_sec': round(throughput, 0),
            'rows_processed':             n_rows,
        })

        print(f'    Run {run}/{runs} → {elapsed:.4f}s | '
              f'CPU: {cpu_pct:.1f}% | '
              f'Mem: {memory_mb:.2f} MB | '
              f'Throughput: {throughput:,.0f} rec/s | '
              f'Rows: {n_rows:,}')

        gc.collect()
        time.sleep(1)   # let system settle between runs

    avg = {
        'method':                     method_name,
        'time_sec':                   round(sum(all_times) / runs, 4),
        'time_min':                   round(min(all_times), 4),
        'time_max':                   round(max(all_times), 4),
        'cpu_percent':                round(sum(all_cpu) / runs, 2),
        'memory_mb':                  round(sum(all_mem) / runs, 2),
        'throughput_records_per_sec': round(sum(all_tput) / runs, 0),
        'rows_processed':             n_rows_fn(result) if n_rows_fn else total_records,
    }

    print(f'    ── Avg: {avg["time_sec"]}s | Min: {avg["time_min"]}s | Max: {avg["time_max"]}s')
    return avg, result


print('✅ Performance measurement framework ready.')

✅ Performance measurement framework ready.


<h2> 4. Pandas Baseline. </h2>

- Read full CSV with no dtype hints (slow type inference)
- Filter jobs that have `salary_min` and `salary_max` listed
- Group by `state` — count postings
- Classify jobs into salary bands using `salary_min`

In [47]:
_ = pd.read_csv(CSV_FILE)
proc.cpu_percent(interval=None)


def pandas_baseline():
    """
    UNOPTIMIZED Pandas pipeline.
    No usecols, no dtype specification, no Categorical.
    """
    # 1. Read entire CSV — no column selection, no dtype hints
    df = pd.read_csv(CSV_FILE)

    # 2. Filter: jobs that have a salary listed
    df_with_salary = df[
        df['salary_min'].notna() & df['salary_max'].notna()
    ].copy()

    # 3. Group by state — count postings
    state_counts = (
        df.groupby('state')['job_id']
        .count()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'job_id': 'count'})
    )

    # 4. Top companies by job count
    top_companies = (
        df.groupby('company')['job_id']
        .count()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'job_id': 'count'})
    )

    # 5. Salary band classification using salary_min (row-by-row, unoptimized)
    def classify_salary(val):
        try:
            v = float(val)
            if v < 2000:    return '< RM 2k'
            elif v < 5000:  return 'RM 2k-5k'
            elif v < 10000: return 'RM 5k-10k'
            else:           return '> RM 10k'
        except (TypeError, ValueError):
            return 'Not Disclosed'

    df['salary_band'] = df['salary_min'].apply(classify_salary)
    salary_dist = df['salary_band'].value_counts()

    return {
          'df': df,
          'df_with_salary': df_with_salary,
          'state_counts': state_counts,
          'top_companies': top_companies,
          'salary_dist': salary_dist,
      }


metrics_baseline, result_baseline = run_benchmark(
    'pandas_baseline',
    pandas_baseline,
    n_rows_fn=lambda r: len(r['df']),
)
all_metrics.append(metrics_baseline)

# Print results
df_b          = result_baseline['df']
state_b       = result_baseline['state_counts']
top_comp_b    = result_baseline['top_companies']
salary_dist_b = result_baseline['salary_dist']

print('=' * 55)
print('  PANDAS BASELINE RESULTS')
print('=' * 55)
print(f'  Total Processing Time : {metrics_baseline["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_baseline["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_baseline["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_baseline["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_b):,}')
print('=' * 55)

print(f'\nJobs with salary listed: {len(result_baseline["df_with_salary"]):,}')

print('\nTop 5 States by Posting Count:')
print(f"  {'State':<30} {'Count':>8}")
for _, row in state_b.head(5).iterrows():
    print(f"  {str(row['state']):<30} {int(row['count']):>8,}")

print('\nTop 5 Companies by Job Count:')
print(f"  {'Company':<30} {'Count':>8}")
for _, row in top_comp_b.head(5).iterrows():
    print(f"  {str(row['company']):<30} {int(row['count']):>8,}")

print('\nSalary Band Distribution (by salary_min):')
for band, cnt in salary_dist_b.items():
    print(f'  {band:<20} : {cnt:,}')

# Save performance_before.csv
pd.DataFrame([metrics_baseline]).to_csv('performance_before.csv', index=False)
print('\n Saved: performance_before.csv')


  Running: pandas_baseline (3 runs)...
    Run 1/3 → 0.9383s | CPU: 76.7% | Mem: 39.38 MB | Throughput: 12,799 rec/s | Rows: 12,010
    Run 2/3 → 1.1265s | CPU: 68.9% | Mem: 39.38 MB | Throughput: 10,661 rec/s | Rows: 12,010
    Run 3/3 → 1.4605s | CPU: 53.7% | Mem: 39.38 MB | Throughput: 8,223 rec/s | Rows: 12,010
    ── Avg: 1.1751s | Min: 0.9383s | Max: 1.4605s
  PANDAS BASELINE RESULTS
  Total Processing Time : 1.1751 seconds
  CPU Usage             : 66.4 %
  Memory Usage          : 39.38 MB
  Throughput            : 10,561 records/second
  Rows Processed        : 12,010

Jobs with salary listed: 6,389

Top 5 States by Posting Count:
  State                             Count
  International                     3,021
  Penang                            1,083
  Selangor                          1,074
  Negeri Sembilan                     959
  Kuala Lumpur                        892

Top 5 Companies by Job Count:
  Company                           Count
  Infineon Technologies    

<h2> 5. Pandas Optimized</h2>

**Optimizations applied:**
1. `usecols` — load only needed columns (saves I/O + RAM)
2. Explicit `dtype` map — skips costly type inference; `salary_min`/`salary_max` loaded directly as `float64`
3. `pd.Categorical` — low-cardinality strings (`state`, `company`, `country`) use integer codes
4. `groupby(observed=True)` — skips empty Categorical combos
5. `pd.cut` with numeric bins — vectorised salary band classification on `salary_min`

In [48]:
_ = pd.read_csv(CSV_FILE)
proc.cpu_percent(interval=None)


def pandas_optimized():
    """
    OPTIMIZED Pandas pipeline.
    1. usecols           — load only 7 needed columns
    2. dtype map         — explicit types, skip inference
    3. pd.Categorical    — job_type, location → integer codes
    4. Vectorised ops    — no apply() for salary parsing
    5. observed=True     — fast groupby with Categorical
    6. pd.cut            — vectorised salary band binning
    """
    NEEDED_COLS = ['job_id', 'title', 'company', 'state',
                   'salary_min', 'salary_max', 'country']

    # Check which columns actually exist
    header_cols = pd.read_csv(CSV_FILE, nrows=0).columns.tolist()
    use_cols = [c for c in NEEDED_COLS if c in header_cols]

    # Opt 1 + 2: usecols + dtype
    df = pd.read_csv(
        CSV_FILE,
        usecols=use_cols,
        dtype={
            'job_id':     'str',
            'title':      'str',
            'company':    'str',
            'state':      'str',
            'country':    'str',
            'salary_min': 'float64',
            'salary_max': 'float64',
        },
        engine='c',
        na_values=['', 'None', 'nan', 'N/A'],
    )

    # Opt 3: Categorical for low-cardinality columns
    for col in ['state', 'company', 'country']:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Filter: jobs with salary listed (vectorised)
    df_with_salary = df[df['salary_min'].notna() & df['salary_max'].notna()]

    # Opt 4: groupby with observed=True
    state_counts = (
        df.groupby('state', observed=True)['job_id']
        .count()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'job_id': 'count'})
    )

    top_companies = (
        df.groupby('company', observed=True)['job_id']
        .count()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'job_id': 'count'})
    )

    # Opt 5: vectorised salary band using pd.cut on salary_min (600 – 20,000)
    df['salary_band'] = pd.cut(
        df['salary_min'],
        bins=[0, 2000, 5000, 10000, 20001],
        labels=['< RM 2k', 'RM 2k-5k', 'RM 5k-10k', '> RM 10k'],
        right=False
    ).cat.add_categories('Not Disclosed').fillna('Not Disclosed')

    salary_dist = df['salary_band'].value_counts()

    return {
        'df': df,
        'df_with_salary': df_with_salary,
        'state_counts': state_counts,
        'top_companies': top_companies,
        'salary_dist': salary_dist,
    }


metrics_optimized, result_optimized = run_benchmark(
    'pandas_optimized',
    pandas_optimized,
    n_rows_fn=lambda r: len(r['df']),
)
all_metrics.append(metrics_optimized)

# Print results
df_o          = result_optimized['df']
state_o       = result_optimized['state_counts']
top_comp_o    = result_optimized['top_companies']
salary_dist_o = result_optimized['salary_dist']

print('=' * 55)
print('  PANDAS OPTIMIZED RESULTS')
print('=' * 55)
print(f'  Total Processing Time : {metrics_optimized["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_optimized["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_optimized["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_optimized["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_o):,}')
print('=' * 55)


print(f'\nJobs with salary listed: {len(result_optimized["df_with_salary"]):,}')

print('\nTop 5 States:')
for _, row in state_o.head(5).iterrows():
    print(f"  {str(row['state']):<30} {int(row['count']):>8,}")

print('\nTop 5 Companies:')
for _, row in top_comp_o.head(5).iterrows():
    print(f"  {str(row['company']):<30} {int(row['count']):>8,}")

print('\nSalary Band Distribution (salary_min):')
for band, cnt in salary_dist_o.items():
    print(f'  {band:<20} : {cnt:,}')


  Running: pandas_optimized (3 runs)...
    Run 1/3 → 1.1982s | CPU: 50.9% | Mem: 3.78 MB | Throughput: 10,024 rec/s | Rows: 12,010
    Run 2/3 → 0.3553s | CPU: 100.5% | Mem: 3.78 MB | Throughput: 33,799 rec/s | Rows: 12,010
    Run 3/3 → 0.3515s | CPU: 101.6% | Mem: 3.78 MB | Throughput: 34,171 rec/s | Rows: 12,010
    ── Avg: 0.635s | Min: 0.3515s | Max: 1.1982s
  PANDAS OPTIMIZED RESULTS
  Total Processing Time : 0.6350 seconds
  CPU Usage             : 84.3 %
  Memory Usage          : 3.78 MB
  Throughput            : 25,998 records/second
  Rows Processed        : 12,010

Jobs with salary listed: 6,389

Top 5 States:
  International                     3,021
  Penang                            1,083
  Selangor                          1,074
  Negeri Sembilan                     959
  Kuala Lumpur                        892

Top 5 Companies:
  Infineon Technologies               185
  McDonald's Malaysia (Gerbang Alaf Restaurants Sdn Bhd)      156
  Club Med                       

<h2> 6. Polars optimization </h2>

In [49]:
proc.cpu_percent(interval=None)   # warm up CPU counter


def polars_pipeline():
    """
    Polars lazy pipeline.
    - scan_csv: lazy, columnar, multi-threaded
    - filter, group_by, agg, sort: deferred until .collect()
    - Polars handles schema inference and parallelism automatically
    """
    lf = pl.scan_csv(
        CSV_FILE,
        # infer_schema_length=10_000,
        null_values=['', 'nan', 'None', 'N/A'],
        schema_overrides={
            'job_id':     pl.Utf8,
            'title':      pl.Utf8,
            'company':    pl.Utf8,
            'state':      pl.Utf8,
            'country':    pl.Utf8,
            'salary_min': pl.Float64,
            'salary_max': pl.Float64,
        },
        ignore_errors=True,
    )

    # Filter: rows with both salary_min and salary_max
    lf_with_salary = lf.filter(
        pl.col('salary_min').is_not_null() & pl.col('salary_max').is_not_null()
    )

    # Group by state
    state_counts = (
        lf
        .group_by('state')
        .agg(pl.count('job_id').alias('count'))
        .sort('count', descending=True)
    )

    # Top companies
    top_companies = (
        lf
        .group_by('company')
        .agg(pl.count('job_id').alias('count'))
        .sort('count', descending=True)
        .head(10)
    )

    # Salary band classification on salary_min (600 – 20,000)
    lf_salary_band = lf.with_columns(
        pl.when(pl.col('salary_min').is_null())
          .then(pl.lit('Not Disclosed'))
          .when(pl.col('salary_min') < 2000)
          .then(pl.lit('< RM 2k'))
          .when(pl.col('salary_min') < 5000)
          .then(pl.lit('RM 2k-5k'))
          .when(pl.col('salary_min') < 10000)
          .then(pl.lit('RM 5k-10k'))
          .otherwise(pl.lit('> RM 10k'))
          .alias('salary_band')
    )

    salary_dist = (
        lf_salary_band
        .group_by('salary_band')
        .agg(pl.count('job_id').alias('count'))
        .sort('salary_band')
    )

# Collect all
    df_full     = lf.collect()
    df_sal      = lf_with_salary.collect()
    df_st       = state_counts.collect()
    df_comp     = top_companies.collect()
    df_sal_dist = salary_dist.collect()

    return {
        'df': df_full,
        'df_with_salary': df_sal,
        'state_counts': df_st,
        'top_companies': df_comp,
        'salary_dist': df_sal_dist,
    }


metrics_polars, result_polars = run_benchmark(
    'polars',
    polars_pipeline,
    n_rows_fn=lambda r: len(r['df']),
)
all_metrics.append(metrics_polars)

df_p      = result_polars['df']
df_st_p   = result_polars['state_counts']
df_comp_p = result_polars['top_companies']
df_sal_p  = result_polars['salary_dist']

print('=' * 55)
print('  POLARS RESULTS')
print('=' * 55)
print(f'  Total Processing Time : {metrics_polars["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_polars["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_polars["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_polars["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {len(df_p):,}')
print('=' * 55)

print(f'\nJobs with salary listed: {len(result_polars["df_with_salary"]):,}')

print('\nTop 5 States:')
for row in df_st_p.head(5).iter_rows(named=True):
    print(f"  {str(row['state']):<30} {int(row['count']):>8,}")

print('\nTop 5 Companies:')
for row in df_comp_p.head(5).iter_rows(named=True):
    print(f"  {str(row['company']):<30} {int(row['count']):>8,}")

print('\nSalary Band Distribution (salary_min):')
for row in df_sal_p.iter_rows(named=True):
    print(f"  {row['salary_band']:<20} : {row['count']:,}")


  Running: polars (3 runs)...
    Run 1/3 → 0.1684s | CPU: 183.7% | Mem: 0.04 MB | Throughput: 71,337 rec/s | Rows: 12,010
    Run 2/3 → 0.2035s | CPU: 147.0% | Mem: 0.04 MB | Throughput: 59,009 rec/s | Rows: 12,010
    Run 3/3 → 0.1611s | CPU: 185.4% | Mem: 0.04 MB | Throughput: 74,528 rec/s | Rows: 12,010
    ── Avg: 0.1777s | Min: 0.1611s | Max: 0.2035s
  POLARS RESULTS
  Total Processing Time : 0.1777 seconds
  CPU Usage             : 172.0 %
  Memory Usage          : 0.04 MB
  Throughput            : 68,291 records/second
  Rows Processed        : 12,010

Jobs with salary listed: 6,389

Top 5 States:
  International                     3,021
  Penang                            1,083
  Selangor                          1,074
  Negeri Sembilan                     959
  Kuala Lumpur                        892

Top 5 Companies:
  Infineon Technologies               185
  McDonald's Malaysia (Gerbang Alaf Restaurants Sdn Bhd)      156
  Club Med                            140
  RHB Ba

<h2> 7. PySpark optimization </h2>

In [50]:
import os
os.environ['PYSPARK_PYTHON'] = 'python3'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python3'

# Build SparkSession ONCE (outside benchmark loop)
print('Starting SparkSession (local[*])...')
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('MaukerjaOptimizationPipeline')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '4')   # keep low for local mode
    .config('spark.ui.showConsoleProgress', 'false')
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true')  # Arrow speedup
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'✅ SparkSession ready — version {spark.version}')
print(f'   Master: {spark.sparkContext.master}')
print(f'   Default parallelism: {spark.sparkContext.defaultParallelism}')

Starting SparkSession (local[*])...
✅ SparkSession ready — version 4.0.2
   Master: local[*]
   Default parallelism: 2


In [51]:
proc.cpu_percent(interval=None)   # warm up

# Warm up Spark with a tiny read so JIT is hot
_ = spark.read.csv(CSV_FILE, header=True, inferSchema=False).limit(10).count()


def pyspark_pipeline():
    """
    PySpark pipeline (local[*] mode).
    - Reads CSV with predefined schema (faster than inferSchema=True)
    - Filters, groups, aggregates using Spark SQL DSL
    - .cache() on the base DataFrame avoids re-reading CSV on each action
    - All actions trigger lazy DAG execution
    """
    schema = StructType([
        StructField('url',         StringType(), True),
        StructField('job_id',      StringType(), True),
        StructField('title',       StringType(), True),
        StructField('company',     StringType(), True),
        StructField('description', StringType(), True),
        StructField('salary_min',  DoubleType(), True),   # 600 – 20,000
        StructField('salary_max',  DoubleType(), True),   # 750 – 41,000
        StructField('state',       StringType(), True),
        StructField('country',     StringType(), True),
    ])

    sdf = (
        spark.read
        .schema(schema)
        .option('header', 'true')
        .option('multiLine', 'true')   # description may contain newlines
        .option('escape', '"')
        .option('nullValue', '')
        .csv(CSV_FILE)
        .cache()                       # persist in memory for reuse
    )

    total_rows = sdf.count()           # trigger caching

    # Filter: jobs with salary_min AND salary_max
    n_with_salary = sdf.filter(
        F.col('salary_min').isNotNull() & F.col('salary_max').isNotNull()
    ).count()

    # Group by state
    state_counts = (
        sdf
        .groupBy('state')
        .agg(F.count('job_id').alias('count'))
        .orderBy(F.desc('count'))
        .limit(10)
        .toPandas()
    )

    # Top companies
    top_companies = (
        sdf
        .groupBy('company')
        .agg(F.count('job_id').alias('count'))
        .orderBy(F.desc('count'))
        .limit(10)
        .toPandas()
    )

    # Salary band classification on salary_min (600 – 20,000)
    sdf_salary = sdf.withColumn(
        'salary_band',
        F.when(F.col('salary_min').isNull(), 'Not Disclosed')
         .when(F.col('salary_min') < 2000,  '< RM 2k')
         .when(F.col('salary_min') < 5000,  'RM 2k-5k')
         .when(F.col('salary_min') < 10000, 'RM 5k-10k')
         .otherwise('> RM 10k')
    )

    salary_dist = (
        sdf_salary
        .groupBy('salary_band')
        .agg(F.count('job_id').alias('count'))
        .orderBy('salary_band')
        .toPandas()
    )

    sdf.unpersist()

    return {
        'total_rows':    total_rows,
        'n_with_salary': n_with_salary,
        'state_counts':  state_counts,
        'top_companies': top_companies,
        'salary_dist':   salary_dist,
    }


metrics_spark, result_spark = run_benchmark(
    'pyspark',
    pyspark_pipeline,
    n_rows_fn=lambda r: r['total_rows'],
)
all_metrics.append(metrics_spark)

# Print results
st_sp   = result_spark['state_counts']
comp_sp = result_spark['top_companies']
sd_sp   = result_spark['salary_dist']

print('=' * 55)
print('  PYSPARK RESULTS')
print('=' * 55)
print(f'  Total Processing Time : {metrics_spark["time_sec"]:.4f} seconds')
print(f'  CPU Usage             : {metrics_spark["cpu_percent"]:.1f} %')
print(f'  Memory Usage          : {metrics_spark["memory_mb"]:.2f} MB')
print(f'  Throughput            : {metrics_spark["throughput_records_per_sec"]:,.0f} records/second')
print(f'  Rows Processed        : {result_spark["total_rows"]:,}')
print('=' * 55)

print(f'\nJobs with salary listed: {result_spark["n_with_salary"]:,}')

print('\nTop 5 States:')
for _, row in st_sp.head(5).iterrows():
    print(f"  {str(row['state']):<30} {int(row['count']):>8,}")

print('\nTop 5 Companies:')
for _, row in comp_sp.head(5).iterrows():
    print(f"  {str(row['company']):<30} {int(row['count']):>8,}")

print('\nSalary Band Distribution (salary_min):')
for _, row in sd_sp.iterrows():
    print(f"  {str(row['salary_band']):<20} : {int(row['count']):,}")

# Stop Spark
# spark.stop()
# print('\n SparkSession stopped.')


  Running: pyspark (3 runs)...
    Run 1/3 → 2.3153s | CPU: 6.0% | Mem: 0.22 MB | Throughput: 5,187 rec/s | Rows: 12,010
    Run 2/3 → 1.4030s | CPU: 10.0% | Mem: 0.21 MB | Throughput: 8,560 rec/s | Rows: 12,010
    Run 3/3 → 1.3867s | CPU: 10.1% | Mem: 0.22 MB | Throughput: 8,661 rec/s | Rows: 12,010
    ── Avg: 1.7017s | Min: 1.3867s | Max: 2.3153s
  PYSPARK RESULTS
  Total Processing Time : 1.7017 seconds
  CPU Usage             : 8.7 %
  Memory Usage          : 0.22 MB
  Throughput            : 7,469 records/second
  Rows Processed        : 12,010

Jobs with salary listed: 6,389

Top 5 States:
  International                     3,021
  Penang                            1,083
  Selangor                          1,074
  Negeri Sembilan                     959
  Kuala Lumpur                        892

Top 5 Companies:
  Infineon Technologies               185
  McDonald's Malaysia (Gerbang Alaf Restaurants Sdn Bhd)      156
  Club Med                            140
  RHB Banking Gr

<h2> 8. Performance comparison tables </h2>

In [52]:
# Build summary DataFrame
df_perf = pd.DataFrame(all_metrics)

baseline_time = df_perf.loc[
    df_perf['method'] == 'pandas_baseline', 'time_sec'
].values[0]

df_perf['speedup_vs_baseline'] = (baseline_time / df_perf['time_sec']).round(2)

# Build per-run log DataFrame
df_runs = pd.DataFrame(all_run_records)

avg_cols = df_perf[
    ['method', 'time_sec', 'cpu_percent', 'memory_mb',
     'throughput_records_per_sec', 'speedup_vs_baseline']
].rename(columns={
    'time_sec':                   'avg_time_sec',
    'cpu_percent':                'avg_cpu_%',
    'memory_mb':                  'avg_memory_mb',
    'throughput_records_per_sec': 'avg_throughput',
})

df_long = df_runs.merge(avg_cols, on='method', how='left')
df_long = df_long.rename(columns={
    'time_sec':                   'time_sec',
    'cpu_percent':                'cpu_%',
    'memory_mb':                  'memory_mb',
    'throughput_records_per_sec': 'throughput',
})

method_order = ['pandas_baseline', 'pandas_optimized', 'polars', 'pyspark']
df_long['method'] = pd.Categorical(
    df_long['method'], categories=method_order, ordered=True
)
df_long = df_long.sort_values(['method', 'run']).reset_index(drop=True)

df_display = df_long[[
    'method', 'run',
    'time_sec', 'cpu_%', 'memory_mb', 'throughput',
    'avg_time_sec', 'avg_cpu_%', 'avg_memory_mb',
    'avg_throughput', 'speedup_vs_baseline'
]].copy()

# Show avg columns only on Run 2 (middle row) — cleaner table
avg_metric_cols = [
    'avg_time_sec', 'avg_cpu_%', 'avg_memory_mb',
    'avg_throughput', 'speedup_vs_baseline'
]
df_clean = df_display.copy()
for col in avg_metric_cols:
    df_clean[col] = [
        v if (i % 3 == 1) else None
        for i, v in enumerate(df_display[col])
    ]

# Save performance_after.csv
df_clean.to_csv('performance_after.csv', index=False)
print('✅ Saved: performance_after.csv')

# Styled display
df_clean_indexed = df_clean.set_index(['method', 'run'])

print(f"\n{'='*95}")
print(f"  FULL BENCHMARK RESULTS — {total_records:,} records | 3 runs each")
print(f"  Avg columns shown on Run 2 (middle row) per method")
print(f"{'='*95}\n")

display(
    df_clean_indexed.style
    .format({
        'time_sec':            '{:.4f}s',
        'cpu_%':               '{:.1f}%',
        'memory_mb':           '{:.2f}',
        'throughput':          '{:,.0f}',
        'avg_time_sec':        '{:.4f}s',
        'avg_cpu_%':           '{:.1f}%',
        'avg_memory_mb':       '{:.2f}',
        'avg_throughput':      '{:,.0f}',
        'speedup_vs_baseline': '{:.2f}x',
    }, na_rep='')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([{
        'selector': 'th',
        'props': [('font-weight', 'bold'), ('background-color', '#e8e8e8'),
                  ('text-align', 'center')]
    }, {
        'selector': 'tr:nth-child(3n)',
        'props': [('border-bottom', '2px solid #aaa')]
    }])
)

# Winner summary
fastest = df_perf.loc[df_perf['time_sec'].idxmin()]
print(f"\n{'='*95}")
print(f"Fastest Method : {fastest['method']}  (avg {fastest['time_sec']}s)")
print(f"Speedup        : {fastest['speedup_vs_baseline']}x faster than pandas baseline")
print(f"Throughput     : {fastest['throughput_records_per_sec']:,.0f} records/sec")
print(f"{'='*95}")

✅ Saved: performance_after.csv

  FULL BENCHMARK RESULTS — 12,010 records | 3 runs each
  Avg columns shown on Run 2 (middle row) per method




Fastest Method : polars  (avg 0.1777s)
Speedup        : 6.61x faster than pandas baseline
Throughput     : 68,291 records/sec
